# Crosscorrelation of Periodic Splines
We select the <span style="color:#c20078">**fuchsia**</span> color to plot the spline $f$ that is cross-correlated with the spline $g,$ in <span style="color:#2ca02c">**green**</span>. The result $f\star g$ of their crosscorrelation is shown as a <span style="color:#1f77b4">**blue**</span> curve, with data samples at the integers represented with circles and stem lines. The sample at the origin, as well as its periodized replicates, is indicated by a red circle and stem line.

In [ ]:
# Load the required libraries
from IPython.display import display
from IPython.display import Math
import ipywidgets as widgets
import math
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 5 # Maximal spline degree
max_delay = 8.0 # Maximal absolute delay

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Initial random periodic cubic splines with Cauchy coefficients
f = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_cauchy(6), degree = 3)
f = f.times(rng.uniform(0.8, 1.2) / math.sqrt(f.variance()))
g = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_cauchy(6), degree = 3)
g = g.times(rng.uniform(0.8, 1.2) / math.sqrt(g.variance()))

# Plot
def update_plot (
    normalized = True,
    period = 6,
    sep1 = "",
    degree_f = 3,
    delay_f = 0.0,
    sep2 = "",
    degree_g = 3,
    delay_g = 0.0
):
    global f
    global g

    # Update of the Cauchy splines
    if f.period != period:
        f = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_cauchy(period),
            degree = f.degree
        )
        f = f.times(rng.uniform(0.8, 1.2) / math.sqrt(f.variance()))
    f.degree = degree_f
    f.delay = delay_f
    if g.period != period:
        g = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_cauchy(period),
            degree = g.degree
        )
        g = g.times(rng.uniform(0.8, 1.2) / math.sqrt(g.variance()))
    g.degree = degree_g
    g.delay = delay_g

    # Crosscorrelation
    r_fg = sk.PeriodicSpline1D.convolve(f.mirrored(), g)
    # Normed crosscorrelation
    rho_fg = sk.PeriodicSpline1D.normed_cross_correlate(f, g)

    # Dynamic range
    image = {f.image(), g.image()}
    plotrange = sk.interval.Interval.enclosure(image)
    plotrange = sk.interval.Closed((
        plotrange.midpoint - 0.55 * plotrange.diameter,
        plotrange.midpoint + 0.55 * plotrange.diameter
    ))

    # Plot of the splines
    (fig, (ax1, ax2)) = plt.subplots(nrows = 2, ncols = 1, sharex = True)
    f.plot(
        (fig, ax1),
        plotrange = plotrange,
        plotpoints = 200 + 1,
        curve_fmt = "#c20078",
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " "
    )
    g.plot(
        (fig, ax1),
        plotrange = plotrange,
        plotpoints = 200 + 1,
        curve_fmt = "-C2",
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " "
    )
    if normalized:
        rho_fg.plot(
            (fig, ax2),
            plotrange = sk.interval.Closed((-1.0, 1.0)),
            plotpoints = 200 + 1,
            knot_marker = " "
        )
    else:
        r_fg.plot(
            (fig, ax2),
            plotpoints = 200 + 1,
            knot_marker = " "
        )
    plt.show()

    # Pearson
    display(Math(
        r"\mbox{{Pearson correlation coefficient: }}\rho_{{fg}}={}"
        .format(rho_fg.at(0))
    ))

# Interaction
normalized_checkbox_widget = widgets.Checkbox(
    value = True,
    description = "Normalized crosscorrelation"
)
widgets.interactive(
    update_plot,
    normalized = normalized_checkbox_widget,
    period = (2, max_period),
    sep1 = widgets.HTML(
        value="<hr style='border:1px solid;margin:15px 0;width:135px'>",
        description = "• • • • •"
    ),
    degree_f = (0, max_degree),
    delay_f = (-max_delay, max_delay),
    sep2 = widgets.HTML(
        value="<hr style='border:1px solid;margin:15px 0;width:135px'>",
        description = "• • • • •"
    ),
    degree_g = (0, max_degree),
    delay_g = (-max_delay, max_delay)
)
